In [1]:
from datasets import load_dataset
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# dataset_name = "squad_v2"
# dataset = load_dataset(dataset_name, split="train")
# eval_dataset = load_dataset(dataset_name, split="validation")
# print("dataset: ",dataset)
# print("eval_dataset: ",eval_dataset)

In [3]:
dataset_path="dataset/small_datset.jsonl"
dataset_name = "tssb_data_3M"
dataset = load_dataset("json", data_files=dataset_path, split="train[:80%]")
eval_dataset = load_dataset("json", data_files=dataset_path, split="train[80%:]")

print("dataset: ",dataset)
print("eval_dataset: ",eval_dataset)

dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 28000
})
eval_dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 7000
})


In [4]:
def combine_prompt_completion(example):
    return {
        "text": example["prompt"] + " " + example["completion"],
        "buggy_code": example["prompt"],
        "fixed_code": example["completion"]
    }

dataset = dataset.map(combine_prompt_completion)
eval_dataset = eval_dataset.map(combine_prompt_completion)
EVAL_REFERENCES = [ex["fixed_code"] for ex in eval_dataset]
print("dataset:", dataset[0],"\n")
print("eval_dataset:", eval_dataset[0],"\n")
print("eval_references:", EVAL_REFERENCES[0],"\n")

dataset: {'prompt': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code:", 'completion': "if 'device_id' in op . metadata and isinstance ( op . metadata [ 'device_id' ] , ( list , tuple ) ) and len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'", 'text': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code: if 'device_id' in op . metadata and isinstance ( op . metadata [ 'device_id' ] , ( list , tuple ) ) and len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'", 'buggy_code': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code:", 'fixed_code': "if 'device_id' in op . metadata and isinsta

In [5]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We    have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


c:\Users\diego\OneDrive\Documentos\Projects\Code-Fixer-LLM-Agent\.venv\Lib\site-packages\unsloth_zoo\gradient_checkpointing.py:330: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"cuda:{i}") for i in range(n_gpus)])


==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.4.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [8]:
import neptune
import neptune.integrations.optuna as optuna_utils

run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)


[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-84


In [9]:
import evaluate
from codebleu import compute_codebleu
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Handle padding/masks
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Extract completions from "##Fixed Code:"
    decoded_completions = []
    for i, pred in enumerate(decoded_preds):
        parts = pred.split("##Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)
        # if i < 2:
        #     print(f"[DEBUG] decoded_pred[{i}]:", repr(pred))
        #     print(f"[DEBUG] completion[{i}]:", repr(completion))
        #     print(f"[DEBUG] reference[{i}]:", repr(EVAL_REFERENCES[i]))

    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])

    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    return {
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy
    }

In [10]:
# from trl import SFTTrainer, SFTConfig
# import time
# start=time.time()

# # SFT Config
# config = SFTConfig(
#     dataset_num_proc = 1,
#     #output_dir="./outputs",
#     dataset_text_field="prompt",#Depends on the colum of your data set
#     #dataset_text_field="question",#Depends on the colum of your data set
#     # learning_rate=1e-5,
#     learning_rate=2e-4,
#     per_device_train_batch_size=3,
#     gradient_accumulation_steps=3,
#     num_train_epochs=5,
#     report_to="none",
#     logging_steps=100,
#     max_steps=1000,
#     eval_accumulation_steps=100,
# )
# trainer = SFTTrainer(
#     model=model,  # base or PEFT model
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     eval_dataset=eval_dataset,
#     args=config,
#     warmup_steps = 5,
#     weight_decay = 0.01,
#     compute_metrics = compute_metrics,
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# metrics = trainer.evaluate()
# print("Metrics:",metrics)
# trainer.train()


# end = time.time()
# length = end - start

# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)

# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")


In [11]:
# metrics = trainer.evaluate()
# print("Metrics:",metrics)


In [12]:
from trl import SFTTrainer, SFTConfig
import time
def objective(trial):
    start=time.time()
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
    num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
    max_steps = trial.suggest_int("max_steps", 500,2000)
    batch_size=2

    # SFT Config
    config = SFTConfig(
        dataset_num_proc = 1,
        #output_dir="./outputs",
        dataset_text_field="text",#Depends on the colum of your data set
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=batch_size,
        num_train_epochs=num_epochs,
        report_to="none",
        logging_steps=100,
        max_steps=max_steps,
        eval_accumulation_steps=100
    )

    trainer = SFTTrainer(
        model=model,  # base or PEFT model
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=eval_dataset,
        args=config,
        warmup_steps = 5,
        weight_decay = 0.01,
        compute_metrics = compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print("Metrics:",metrics)

    # Log trial info to Neptune
    run[f"trial/{trial.number}/metrics"] = metrics
    run[f"trial/{trial.number}/params"] = {
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "max_steps": max_steps,
    }


    end = time.time()
    length = end - start

    hours = int(length // 3600)
    minutes = int((length % 3600) // 60)
    seconds = int(length % 60)

    print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

    return metrics["eval_loss"]  # Or any other metric

In [13]:
import optuna
neptune_callback = optuna_utils.NeptuneCallback(run)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5, callbacks=[neptune_callback], show_progress_bar=True)

[I 2025-05-17 23:47:39,315] A new study created in memory with name: no-name-c9e9c60e-5e03-48ba-a5c4-9c1d74e049d1


  0%|          | 0/5 [00:00<?, ?it/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/7000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,000 | Num Epochs = 1 | Total steps = 1,131
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Step,Training Loss
100,1.212200
200,1.101300
300,1.089600
400,1.072700
500,1.071800
600,1.013700
700,1.034200
800,1.009500
900,0.994800
1000,0.965800


...

Step,Training Loss
100,0.874600
200,3.174100
300,7.179900
400,6.625700
500,6.334600
600,5.813400
700,5.293800
800,4.749100
900,4.616900
1000,4.481400


Metrics: {'eval_loss': 4.0175347328186035, 'eval_codebleu': 0.7128105975966934, 'eval_bleu': 0.007122000908955168, 'eval_precisions': [0.28889020455285513, 0.02935437544878399, 0.0036483999042501384, 0.0002626487248404409], 'eval_brevity_penalty': 0.7501207452539341, 'eval_length_ratio': 0.7766863055036644, 'eval_translation_length': 135124, 'eval_reference_length': 173975, 'eval_rouge1': 0.08164671186960376, 'eval_rouge2': 0.004340039219784472, 'eval_rougeL': 0.07896011791013159, 'eval_rougeLsum': 0.07913261240965361, 'eval_accuracy': 0.4052217983798137, 'eval_runtime': 204.9105, 'eval_samples_per_second': 34.161, 'eval_steps_per_second': 8.54}
It took 0 hours, 23 minutes, and 56 seconds to train the model!
[I 2025-05-18 01:04:40,756] Trial 3 finished with value: 4.0175347328186035 and parameters: {'learning_rate': 0.0004868349751940819, 'num_train_epochs': 6, 'max_steps': 1470}. Best is trial 0 with value: 1.0051050186157227.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,000 | Num Epochs = 1 | Total steps = 1,720
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Step,Training Loss
100,4.097000
200,4.008800
300,4.041900
400,4.000400
500,4.118400
600,3.943500
700,4.057300
800,3.953900
900,4.015700
1000,3.981500


Metrics: {'eval_loss': 3.917722702026367, 'eval_codebleu': 0.7055238662580442, 'eval_bleu': 0.008929427449972417, 'eval_precisions': [0.3026143116235863, 0.035339479780469224, 0.004495915046109841, 0.00040956821053548864], 'eval_brevity_penalty': 0.7537895717619629, 'eval_length_ratio': 0.7796407529817503, 'eval_translation_length': 135638, 'eval_reference_length': 173975, 'eval_rouge1': 0.09117100815446968, 'eval_rouge2': 0.005153587390190393, 'eval_rougeL': 0.0882411050818504, 'eval_rougeLsum': 0.08838427332630944, 'eval_accuracy': 0.4133496879591314, 'eval_runtime': 175.3056, 'eval_samples_per_second': 39.93, 'eval_steps_per_second': 9.983}
It took 0 hours, 19 minutes, and 56 seconds to train the model!
[I 2025-05-18 01:24:37,278] Trial 4 finished with value: 3.917722702026367 and parameters: {'learning_rate': 4.8573052722769554e-05, 'num_train_epochs': 8, 'max_steps': 1720}. Best is trial 0 with value: 1.0051050186157227.


In [14]:
# Get the best parameters
best_trial = study.best_trial

best_params = best_trial.params
print("best_params: ",best_params)

best_value = best_trial.value
print("Eval loss:", best_value)
run.stop()

best_params:  {'learning_rate': 0.00016215142122972796, 'num_train_epochs': 10, 'max_steps': 1131}
Eval loss: 1.0051050186157227
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 60 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 60 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/casvi/CodeMedic/e/COD-84/metadata
